In [1]:
import gurobipy as gp
from gurobipy import GRB

In [2]:
width = 4
height = 4

In [3]:
# MUST BE OF LENGTH MN-1
tiles = ["A", "A", "B", "B", "C", "C", "D", "D", "E", "E", "H", "I", "J", "K", "K"]

In [4]:
tiles.insert(0, "X") # THE EMPTY TILE IS ALWAYS GIVEN INDEX 0

In [5]:
faces_up = [i for i in range(width * height) if (tiles[i] in ["B", "E", "F", "G", "H", "J", "K"])]
faces_down = [i for i in range(width * height) if (tiles[i] in ["C", "D", "F", "G", "H", "J", "K"])]
faces_left = [i for i in range(width * height) if (tiles[i] in ["B", "C", "F", "G", "I", "J", "K"])]
faces_right = [i for i in range(width * height) if (tiles[i] in ["D", "E", "F", "G", "I", "J", "K"])]

In [6]:
not_faces_up = [i for i in range(width * height) if (i not in faces_up)]
not_faces_down = [i for i in range(width * height) if (i not in faces_down)]
not_faces_left = [i for i in range(width * height) if (i not in faces_left)]
not_faces_right = [i for i in range(width * height) if (i not in faces_right)]

In [7]:
model = gp.Model("mip1")

Set parameter TokenServer to value "gurobilicense.davidson.edu"


In [8]:
decision_vars = model.addVars(width * height, width * height, vtype=GRB.BINARY, name="decision_vars")

In [9]:
model.addConstrs((decision_vars.sum(i, "*") == 1 for i in range(width * height)), name="One Of Each Tile")

{0: <gurobi.Constr *Awaiting Model Update*>,
 1: <gurobi.Constr *Awaiting Model Update*>,
 2: <gurobi.Constr *Awaiting Model Update*>,
 3: <gurobi.Constr *Awaiting Model Update*>,
 4: <gurobi.Constr *Awaiting Model Update*>,
 5: <gurobi.Constr *Awaiting Model Update*>,
 6: <gurobi.Constr *Awaiting Model Update*>,
 7: <gurobi.Constr *Awaiting Model Update*>,
 8: <gurobi.Constr *Awaiting Model Update*>,
 9: <gurobi.Constr *Awaiting Model Update*>,
 10: <gurobi.Constr *Awaiting Model Update*>,
 11: <gurobi.Constr *Awaiting Model Update*>,
 12: <gurobi.Constr *Awaiting Model Update*>,
 13: <gurobi.Constr *Awaiting Model Update*>,
 14: <gurobi.Constr *Awaiting Model Update*>,
 15: <gurobi.Constr *Awaiting Model Update*>}

In [10]:
model.addConstrs((decision_vars.sum("*", j) == 1 for j in range(width * height)), name="No Overlapping Tiles")

{0: <gurobi.Constr *Awaiting Model Update*>,
 1: <gurobi.Constr *Awaiting Model Update*>,
 2: <gurobi.Constr *Awaiting Model Update*>,
 3: <gurobi.Constr *Awaiting Model Update*>,
 4: <gurobi.Constr *Awaiting Model Update*>,
 5: <gurobi.Constr *Awaiting Model Update*>,
 6: <gurobi.Constr *Awaiting Model Update*>,
 7: <gurobi.Constr *Awaiting Model Update*>,
 8: <gurobi.Constr *Awaiting Model Update*>,
 9: <gurobi.Constr *Awaiting Model Update*>,
 10: <gurobi.Constr *Awaiting Model Update*>,
 11: <gurobi.Constr *Awaiting Model Update*>,
 12: <gurobi.Constr *Awaiting Model Update*>,
 13: <gurobi.Constr *Awaiting Model Update*>,
 14: <gurobi.Constr *Awaiting Model Update*>,
 15: <gurobi.Constr *Awaiting Model Update*>}

In [11]:
top_indices = [i for i in range(width)]
for slot in top_indices:
    for tile in faces_up:
        model.addConstr(decision_vars[tile, slot] == 0)

In [12]:
bottom_indices = [i + width * height - width for i in range(width)]
for slot in bottom_indices:
    for tile in faces_down:
        model.addConstr(decision_vars[tile, slot] == 0)

In [13]:
left_indices = [i * width for i in range(height)]
for slot in left_indices:
    for tile in faces_left:
        model.addConstr(decision_vars[tile, slot] == 0)

In [14]:
right_indices = [i * width + width - 1 for i in range(height)]
for slot in right_indices:
    for tile in faces_right:
        model.addConstr(decision_vars[tile, slot] == 0)

In [15]:
for row in range(height):
    for column in range(width - 1):
        slot = row * width + column
        for tileLeft in faces_right:
            for tileRight in not_faces_left:
                model.addConstr(decision_vars[tileLeft, slot] + decision_vars[tileRight, slot + 1] <= 1)

In [16]:
for row in range(height):
    for column in range(1, width):
        slot = row * width + column
        for tileRight in faces_left:
            for tileLeft in not_faces_right:
                model.addConstr(decision_vars[tileRight, slot] + decision_vars[tileLeft, slot - 1] <= 1)

In [17]:
for column in range(width):
    for row in range(height - 1):
        slot = row * width + column
        for tileTop in faces_down:
            for tileBottom in not_faces_up:
                model.addConstr(decision_vars[tileTop, slot] + decision_vars[tileBottom, slot + width] <= 1)

In [18]:
for column in range(width):
    for row in range(1, height):
        slot = row * width + column
        for tileBottom in faces_up:
            for tileTop in not_faces_down:
                model.addConstr(decision_vars[tileBottom, slot] + decision_vars[tileTop, slot - width] <= 1)

In [19]:
model.setParam('OutputFlag', 0) # OTHERWISE YOU'LL GET HUNDREDS OF OUTPUT PRINTS

In [20]:
sols = []
model.optimize()
while (model.status != 3): # ALSO KNOWN AS GRB.INFEASIBLE, BUT THAT DIDN'T WORK FOR SOME REASON
    sol = [0] * (height * width)
    for tile in range(width * height):
        for row in range(height):
            for column in range(width):
                slot = row * width + column
                if (decision_vars[tile, slot].X == 1):
                    sol[slot] = tiles[tile]
    if sol not in sols:
        sols.append(sol)
        print(sol)
    active_vars = [var for var in model.getVars() if var.x == 1]
    model.addConstr(sum(active_vars) <= width * height - 1) # GURANTEES PREVIOUS SOLUTION WILL NO LONGER BE FOUND
    model.reset()
    model.optimize()
print(len(sols))

['A', 'D', 'C', 'A', 'D', 'K', 'K', 'C', 'E', 'J', 'B', 'H', 'X', 'E', 'I', 'B']
['A', 'D', 'C', 'X', 'D', 'K', 'K', 'C', 'E', 'J', 'B', 'H', 'A', 'E', 'I', 'B']
['D', 'I', 'C', 'A', 'H', 'D', 'K', 'C', 'E', 'K', 'J', 'B', 'X', 'E', 'B', 'A']
['X', 'D', 'C', 'A', 'D', 'J', 'K', 'C', 'E', 'K', 'B', 'H', 'A', 'E', 'I', 'B']
['A', 'D', 'C', 'X', 'D', 'J', 'K', 'C', 'H', 'E', 'K', 'B', 'E', 'I', 'B', 'A']
['A', 'D', 'I', 'C', 'D', 'K', 'C', 'H', 'E', 'K', 'J', 'B', 'A', 'E', 'B', 'X']
['D', 'I', 'C', 'A', 'H', 'D', 'K', 'C', 'E', 'J', 'K', 'B', 'X', 'E', 'B', 'A']
['D', 'I', 'C', 'A', 'H', 'D', 'J', 'C', 'E', 'K', 'K', 'B', 'X', 'E', 'B', 'A']
['X', 'D', 'I', 'C', 'D', 'K', 'C', 'H', 'E', 'J', 'K', 'B', 'A', 'E', 'B', 'A']
['X', 'D', 'C', 'A', 'D', 'K', 'J', 'C', 'H', 'E', 'K', 'B', 'E', 'I', 'B', 'A']
['X', 'D', 'C', 'A', 'D', 'K', 'K', 'C', 'E', 'J', 'B', 'H', 'A', 'E', 'I', 'B']
['A', 'D', 'C', 'X', 'D', 'J', 'K', 'C', 'E', 'K', 'B', 'H', 'A', 'E', 'I', 'B']
['A', 'D', 'C', 'A', 'D', 'K